# Figure 5.9: Zooming into a spectrum

*The Discrete Algebra of the Fourier Transform*, Gabriel Peyré.

A noisy cosine, its coarse spectrum, a local spectral zoom, and an adjusted spectrum after shortening the record using the estimated peak. Spectral magnitudes are divided by the respective record lengths. The generating frequency is distinguished from its estimate.

[Read this figure in the book](https://mathematical-tours.github.io/daft-sources/DAFT-EN.pdf#page=123).

## Prepare the example

Run the cells in order in Jupyter or Google Colab. The first cell installs missing Python packages and downloads the figure sources and input data once. Subsequent runs use the cached download. Python 3.10 or later is required.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import urllib.request
import zipfile

packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "PIL": "Pillow", "sympy": "sympy"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

archive_url = os.environ.get("DAFT_FIGURE_ARCHIVE", 'https://mathematical-tours.github.io/daft-sources/downloads/daft-figures-2e843bbe89f0.zip')
archive_sha = '2e843bbe89f0e274849b6896eb16bd9f7686521a8ca98f633240bfc06ea2d5b1'
workspace = Path(os.environ.get("DAFT_FIGURE_WORKDIR", "daft-figure-files")) / archive_sha[:12]
cache = workspace / "sources.zip"
workspace.mkdir(parents=True, exist_ok=True)
data = cache.read_bytes() if cache.exists() else urllib.request.urlopen(archive_url, timeout=60).read()
if hashlib.sha256(data).hexdigest() != archive_sha:
    raise ValueError("The figure-source download is incomplete or has changed. Download it again.")
cache.write_bytes(data)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(workspace)
ROOT = (workspace / "daft-figures").resolve()
__file__ = str(ROOT / 'figures-code/chapter-related-transforms/spectral-zoom-and-adjustment/generate.py')
print("Figure sources ready.")


## Generate the figure

This is the plotting code used by the book. Change its parameters and rerun this cell, then the display cell below, to explore the example.

In [ ]:
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[3]
sys.path.insert(0,str(ROOT/"tools"))
sys.path.insert(0,str(ROOT/"listings"))
from figure_style import *
from chirp import zoom_spectrum,scaled_fourier
n=128;beta=2.23;t=np.arange(n);f=np.cos(2*np.pi*beta*t/n)+.2*(np.random.default_rng(52).random(n)-.5)
nu=np.linspace(1.7,2.8,1001);zoom=zoom_spectrum(f,nu[0],nu[1]-nu[0],nu.size)
estimate=nu[np.argmax(abs(zoom))];b=int(np.floor(estimate));L=n*b/estimate;r=round(L)
adjusted=scaled_fourier(f[:r],r/L)
fig,axes=plt.subplots(2,2,figsize=(9,6));a,bax,c,d=axes.ravel()
a.plot(t,f,color=TEAL);a.set(title='(a) Observed signal',xlim=(0,n-1));axes_box(a,'sample index $n$','amplitude')
k=np.arange(32);bax.plot(k,abs(np.fft.fft(f)[:32])/n,'o-',ms=3,color=TEAL);bax.set(title='(b) Coarse spectrum',xlim=(0,31));axes_box(bax,'cycles per original record','magnitude / $N$')
c.plot(nu,abs(zoom)/n,color=TEAL);c.axvline(beta,color=RED,ls=':',label=rf'true $\beta={beta}$')
c.axvline(estimate,color=BLUE,ls='--',label=rf'peak $\widehat\beta={estimate:.3f}$');c.set(title='(c) Spectral zoom',xlim=(nu[0],nu[-1]));c.legend(fontsize=8,loc='lower left');axes_box(c,'cycles per original record','magnitude / $N$')
d.plot(k,abs(adjusted[:32])/r,'o-',ms=3,color=TEAL);d.set(title=rf'(d) Adjusted record: $r={r}$',xlim=(0,31));axes_box(d,'adjusted frequency index $k$','magnitude / $r$')
for ax in [bax,d]:ax.set_ylim(-.012,.58)
save(fig,'chapter-related-transforms','spectral-zoom-and-adjustment')


## Display and save

In [ ]:
from IPython.display import Image, FileLink, display

display(Image(filename=str(ROOT / 'build/figure-previews/chapter-related-transforms/spectral-zoom-and-adjustment.png')))
display(FileLink(os.path.relpath(ROOT / 'figures/chapter-related-transforms/spectral-zoom-and-adjustment/figure.pdf', Path.cwd()),
                 result_html_prefix="Download the vector PDF: "))
